# Imports

In [27]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings
warnings.filterwarnings('ignore')

# Create Clothes Dataset

In [295]:
clothes_id   = random.sample(range(1, 100000), 25) # Gets a random, unique sample of 25 different numbers from the range given
color        = [random.choice(['Blue', 'Red', 'Green', 'Purple' , 'Black']) for _ in range(25)] # 25 Random choices between the colors
size         = [random.choice(['S', 'M', 'L', 'XL', 'XXL']) for _ in range(25)] # 25 Random Choices between sizes
clothes_type = [random.choice(['Shirt', 'Pants', 'Jacket', 'Shorts']) for _ in range(25)] # 25 Random Choices between clothes type
class_type   = [random.choice(['class1', 'class2', 'class3', 'class4']) for _ in range(25)] # Target Variable

clothes_dict = {
    'ID': clothes_id,
    'Color': color,
    'Size': size,
    'Type': clothes_type,
    'Class': class_type
}

clothes_df = pd.DataFrame(data=clothes_dict)

X = clothes_df.iloc[:, :-1].values
y = clothes_df.iloc[:, -1].values

# Explore Data

In [261]:
cat_cols = clothes_df.select_dtypes('object').columns.tolist()
cat_cols

['Color', 'Size', 'Type', 'Class']

In [262]:
print(f'Value Counts for Color field: {clothes_df['Color'].value_counts()}')
print(f'\nValue Counts for Size field: {clothes_df['Size'].value_counts()}')
print(f'\nValue Counts for Type field: {clothes_df['Type'].value_counts()}')

Value Counts for Color field: Color
Green     6
Black     6
Purple    6
Blue      5
Red       2
Name: count, dtype: int64

Value Counts for Size field: Size
M      7
XL     6
XXL    6
S      4
L      2
Name: count, dtype: int64

Value Counts for Type field: Type
Shorts    8
Shirt     7
Pants     6
Jacket    4
Name: count, dtype: int64


For the Color field, we can use One Hot Encoder because it is nominal data

For the Size field, we need to use Ordinal Encoder because it is ordinal data

For the Type field, we need to use One Hot Encoder as well

# Encode Data

## Ordinal Feature Encoding (Size)

### Manual Mapping

In [263]:
clothes_df_copy = clothes_df.copy()
# Color Feature
clothes_df_copy['Size'].unique()

# Color Mapping
size_map = {
    'S': 1,
    'M': 2,
    'L': 3,
    'XL': 4, 
    'XXL': 5
}

clothes_df_copy['Size'] = clothes_df_copy['Size'].map(size_map)

clothes_df_copy.head(2)

,ID,Color,Size,Type,Class
0,88043,Red,4,Shorts,class4
1,50637,Green,4,Pants,class4


If you need to go back to the original label feature values (the categories) you can do inverse mappnig

In [264]:
inv_size_mapping = {v: k for k, v in size_map.items()}

clothes_df_copy['Size'] = clothes_df_copy['Size'].map(inv_size_mapping)

clothes_df_copy.head()

,ID,Color,Size,Type,Class
0,88043,Red,XL,Shorts,class4
1,50637,Green,XL,Pants,class4
2,44630,Black,XL,Shorts,class4
3,3088,Blue,S,Shirt,class1
4,70827,Blue,S,Shorts,class2


In [265]:
# Put it back to encoded data
clothes_df_copy['Size'] = clothes_df_copy['Size'].map(size_map)

clothes_df_copy.head(2)

,ID,Color,Size,Type,Class
0,88043,Red,4,Shorts,class4
1,50637,Green,4,Pants,class4


### OrdinalEncoder() with NumPy

In [266]:
X[:5, :]

array([[88043, 'Red', 'XL', 'Shorts'],
       [50637, 'Green', 'XL', 'Pants'],
       [44630, 'Black', 'XL', 'Shorts'],
       [3088, 'Blue', 'S', 'Shirt'],
       [70827, 'Blue', 'S', 'Shorts']], dtype=object)

In [267]:
from sklearn.preprocessing import OrdinalEncoder

custom_order = [['S', 'M', 'L', 'XL', 'XXL']] # Create the order of the labesl
ord_encoder = OrdinalEncoder(categories=custom_order)

# Change the third column (Size)
X[:, 2] = ord_encoder.fit_transform(X[:, 2].reshape(-1, 1)).ravel() # Encode the Size values using the encoder and numpy. Reshape adds a column to be able to run the encoder, and ravel turns it back into a vector

X[:5, :]

array([[88043, 'Red', 3.0, 'Shorts'],
       [50637, 'Green', 3.0, 'Pants'],
       [44630, 'Black', 3.0, 'Shorts'],
       [3088, 'Blue', 0.0, 'Shirt'],
       [70827, 'Blue', 0.0, 'Shorts']], dtype=object)

### OrdinalEncoder() with Pandas

In [268]:
clothes_df.head(3)

,ID,Color,Size,Type,Class
0,88043,Red,XL,Shorts,class4
1,50637,Green,XL,Pants,class4
2,44630,Black,XL,Shorts,class4


In [269]:
clothes_df['Size'].unique().tolist()

['XL', 'S', 'M', 'XXL', 'L']

In [270]:
ord_encoder2 = OrdinalEncoder(categories=[['S', 'M', 'L', 'XL', 'XXL']])

clothes_df['Size'] = ord_encoder2.fit_transform(clothes_df[['Size']])

clothes_df.head()

,ID,Color,Size,Type,Class
0,88043,Red,3.0,Shorts,class4
1,50637,Green,3.0,Pants,class4
2,44630,Black,3.0,Shorts,class4
3,3088,Blue,0.0,Shirt,class1
4,70827,Blue,0.0,Shorts,class2


## Nominal Feature Encoding (Color, Type)

### Using get_dummies() method from Pandas

In [271]:
clothes_df_copy.head()

,ID,Color,Size,Type,Class
0,88043,Red,4,Shorts,class4
1,50637,Green,4,Pants,class4
2,44630,Black,4,Shorts,class4
3,3088,Blue,1,Shirt,class1
4,70827,Blue,1,Shorts,class2


In [272]:
clothes_df_copy = pd.get_dummies(clothes_df_copy, columns=['Color']) 
clothes_df_copy = clothes_df_copy.drop(columns='Color_Red')
clothes_df_copy.head()

,ID,Size,Type,Class,Color_Black,Color_Blue,Color_Green,Color_Purple
0,88043,4,Shorts,class4,False,False,False,False
1,50637,4,Pants,class4,False,False,True,False
2,44630,4,Shorts,class4,True,False,False,False
3,3088,1,Shirt,class1,False,True,False,False
4,70827,1,Shorts,class2,False,True,False,False


### Using OneHotEncoder() and NumPy

In [273]:
X[:5, :]

array([[88043, 'Red', 3.0, 'Shorts'],
       [50637, 'Green', 3.0, 'Pants'],
       [44630, 'Black', 3.0, 'Shorts'],
       [3088, 'Blue', 0.0, 'Shirt'],
       [70827, 'Blue', 0.0, 'Shorts']], dtype=object)

In [274]:
from sklearn.preprocessing import OneHotEncoder

# To avoid multicollinearity, which can happen when encoding, we want to drop the first column using drop='first'
onehot_encoder = OneHotEncoder(drop='first')

encoded_colors = onehot_encoder.fit_transform(X[:, 1].reshape(-1, 1)).toarray() # We do toarray() for this because OneHotEncoder produces a sparse matrix, but we want a regular numpy matrix

# Drop old Color column (index = 1) and concatenate the new color vectors in that spot
X = np.concatenate((X[:, :1], encoded_colors, X[:, 2:]), axis=1)

X

array([[88043, 0.0, 0.0, 0.0, 1.0, 3.0, 'Shorts'],
       [50637, 0.0, 1.0, 0.0, 0.0, 3.0, 'Pants'],
       [44630, 0.0, 0.0, 0.0, 0.0, 3.0, 'Shorts'],
       [3088, 1.0, 0.0, 0.0, 0.0, 0.0, 'Shirt'],
       [70827, 1.0, 0.0, 0.0, 0.0, 0.0, 'Shorts'],
       [43206, 0.0, 0.0, 1.0, 0.0, 1.0, 'Shirt'],
       [79277, 1.0, 0.0, 0.0, 0.0, 1.0, 'Jacket'],
       [26204, 0.0, 1.0, 0.0, 0.0, 1.0, 'Pants'],
       [5449, 0.0, 1.0, 0.0, 0.0, 4.0, 'Shirt'],
       [97106, 0.0, 1.0, 0.0, 0.0, 4.0, 'Pants'],
       [22727, 0.0, 1.0, 0.0, 0.0, 4.0, 'Shirt'],
       [30991, 0.0, 0.0, 0.0, 0.0, 0.0, 'Shorts'],
       [26958, 0.0, 0.0, 0.0, 0.0, 2.0, 'Jacket'],
       [582, 1.0, 0.0, 0.0, 0.0, 1.0, 'Jacket'],
       [13017, 0.0, 0.0, 1.0, 0.0, 3.0, 'Pants'],
       [17588, 0.0, 0.0, 0.0, 0.0, 4.0, 'Shorts'],
       [63711, 1.0, 0.0, 0.0, 0.0, 3.0, 'Shirt'],
       [68076, 0.0, 0.0, 0.0, 0.0, 4.0, 'Pants'],
       [57582, 0.0, 0.0, 1.0, 0.0, 1.0, 'Shirt'],
       [85714, 0.0, 0.0, 0.0, 1.0, 1.0, 'Pants

### Using OneHotEncoder() with Pandas

In [275]:
clothes_df_copy.head(1)

,ID,Size,Type,Class,Color_Black,Color_Blue,Color_Green,Color_Purple
0,88043,4,Shorts,class4,False,False,False,False


In [276]:
clothes_df.head(1)

,ID,Color,Size,Type,Class
0,88043,Red,3.0,Shorts,class4


In [277]:
# Drop the first column for each encoding 
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first').set_output(transform='pandas')

ohe_transform = ohe.fit_transform(clothes_df[['Color', 'Type']])

clothes_df = pd.concat([clothes_df, ohe_transform], axis=1).drop(columns=['Color', 'Type'])

clothes_df.head()

,ID,Size,Class,Color_Blue,Color_Green,Color_Purple,Color_Red,Type_Pants,Type_Shirt,Type_Shorts
0,88043,3.0,class4,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,50637,3.0,class4,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,44630,3.0,class4,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3088,0.0,class1,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,70827,0.0,class2,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## Label/Target Feature Encoding (Class)

### Using LabelEncoder() with NumPy

In [278]:
clothes_df.head()

,ID,Size,Class,Color_Blue,Color_Green,Color_Purple,Color_Red,Type_Pants,Type_Shirt,Type_Shorts
0,88043,3.0,class4,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,50637,3.0,class4,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,44630,3.0,class4,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3088,0.0,class1,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,70827,0.0,class2,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [279]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

print(f'X Matrix: \n{X[:5, :]}')
print('')
print(f'Y Vector: \n{y[:5]}')

X Matrix: 
[[88043 0.0 0.0 0.0 1.0 3.0 'Shorts']
 [50637 0.0 1.0 0.0 0.0 3.0 'Pants']
 [44630 0.0 0.0 0.0 0.0 3.0 'Shorts']
 [3088 1.0 0.0 0.0 0.0 0.0 'Shirt']
 [70827 1.0 0.0 0.0 0.0 0.0 'Shorts']]

Y Vector: 
[3 3 3 0 1]


### Using LabelEncoder() with Pandas

In [281]:
clothes_df.head()

,ID,Size,Class,Color_Blue,Color_Green,Color_Purple,Color_Red,Type_Pants,Type_Shirt,Type_Shorts
0,88043,3.0,class4,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,50637,3.0,class4,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,44630,3.0,class4,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3088,0.0,class1,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,70827,0.0,class2,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [282]:
label_encoder = LabelEncoder()

clothes_df['Class'] = label_encoder.fit_transform(clothes_df['Class'])

clothes_df.head()

,ID,Size,Class,Color_Blue,Color_Green,Color_Purple,Color_Red,Type_Pants,Type_Shirt,Type_Shorts
0,88043,3.0,3,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,50637,3.0,3,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,44630,3.0,3,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,3088,0.0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,70827,0.0,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## Full Dataframe Encoding using ColumnTransformer

Need to re-run the dataframe clothes_df to run the ColumnTransformer method

In [291]:
clothes_df.head()

,ID,Color,Size,Type,Class
0,16595,Green,XXL,Jacket,class3
1,22512,Red,XXL,Pants,class4
2,10156,Green,S,Jacket,class3
3,71650,Purple,S,Pants,class3
4,72206,Blue,S,Jacket,class3


### ColumnTransformer NumPy Version
- Pass the column indices in instead of column names

In [296]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), [1, 3]),
        ('ordinal', OrdinalEncoder(categories=[['S', 'M', 'L', 'XL', 'XXL']]), [2])
    ],
    remainder='passthrough'
)

X = ct.fit_transform(X)

In [297]:
X[:5]

array([[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 2.0, 72085],
       [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 93036],
       [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 22011],
       [1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 2.0, 10547],
       [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 2.0, 23953]],
      dtype=object)

### ColumnTransformer Pandas Version
 - Pass the column names in instead of the column indices

In [308]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import make_column_transformer

# Column Transforming for the X Features
ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['Color', 'Type']),
        ('ordinal', OrdinalEncoder(categories=[['S', 'M', 'L', 'XL', 'XXL']]), ['Size']),
    ],
    remainder='passthrough'
)

clothes_df_ct = pd.DataFrame(data=ct.fit_transform(clothes_df), columns=ct.get_feature_names_out())
print(clothes_df_ct.columns)
# Label Encoding for the y Feature (target) after transforming X features
label_encoder = LabelEncoder()
clothes_df_ct['remainder__Class'] = label_encoder.fit_transform(clothes_df_ct['remainder__Class'])

clothes_df_ct.head()

Index(['onehot__Color_Black', 'onehot__Color_Blue', 'onehot__Color_Green',
       'onehot__Color_Purple', 'onehot__Color_Red', 'onehot__Type_Jacket',
       'onehot__Type_Pants', 'onehot__Type_Shirt', 'onehot__Type_Shorts',
       'ordinal__Size', 'remainder__ID', 'remainder__Class'],
      dtype='str')


,onehot__Color_Black,onehot__Color_Blue,onehot__Color_Green,onehot__Color_Purple,onehot__Color_Red,onehot__Type_Jacket,onehot__Type_Pants,onehot__Type_Shirt,onehot__Type_Shorts,ordinal__Size,remainder__ID,remainder__Class
0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,72085,3
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,93036,0
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,22011,3
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,2.0,10547,3
4,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,23953,3


These are some techniques to encode categorical data before passing to a Machine Leanring model. It is import to conduct this preprocessing step before training a model because most models need to digest numerical data to be accurate.